# 01 — Captura de stream (Fase 0)

**Objetivo:** Validar ingesta desde webcam, archivo `.mp4` o RTSP con OpenCV.

| Entrada | Salida |
|---------|--------|
| `SOURCE` (auto / path / 0 / rtsp://…) | `outputs/01_capture/frame_%06d.jpg` + `metadata.json` |

**Criterio de éxito:** ≥30 frames guardados sin errores de lectura consecutivos.


## Prerrequisitos

- `uv sync --all-groups`
- Opcional: clip InHARD en `data_sample/InHARD-master/01-InHARD/Segmented/RGBSegmented/`
- Kernel: Python 3.11 del proyecto (`.venv`)


## 1. Setup


In [9]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import cv2

from _common.io import (
    ensure_scripts_on_path,
    load_dotenv_repo,
    repo_root,
    resolve_source_video,
    setup_logging,
    stage_output_dir,
    utc_now_iso,
    write_json,
)
from loguru import logger

ensure_scripts_on_path()
load_dotenv_repo()
setup_logging()
logger.info("Repo root: {}", repo_root())


22:02:38 | INFO | Repo root: /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve


## 2. Configuration


In [4]:
# --- Editar estas variables ---
SOURCE = "auto"  # "auto" | ruta .mp4 | 0 (webcam) | "rtsp://..."
MAX_FRAMES = 120
FPS_SAMPLE = 2.0  # guardar ~1 frame cada 1/FPS_SAMPLE segundos (aprox.)
OUT_DIR = stage_output_dir("01_capture")
MIN_FRAMES_OK = 30


## 3. Captura y muestreo


In [5]:
resolved = resolve_source_video(SOURCE)
cap = cv2.VideoCapture(resolved)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir la fuente: {resolved!r}")

fps_native = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_interval = max(1, int(round(fps_native / max(FPS_SAMPLE, 0.1))))

saved = 0
frame_idx = 0
timestamps: list[float] = []
t0 = time.time()

while saved < MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        logger.warning("Fin de stream o lectura fallida en frame {}", frame_idx)
        break
    if frame_idx % frame_interval == 0:
        out_path = OUT_DIR / f"frame_{saved:06d}.jpg"
        cv2.imwrite(str(out_path), frame)
        timestamps.append(time.time() - t0)
        saved += 1
    frame_idx += 1

cap.release()
logger.info("Frames guardados: {}", saved)


22:02:09 | INFO | Using InHARD clip: /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/data_sample/InHARD-master/01-InHARD/Segmented/RGBSegmented/Assemble system/P01_R01_0013.84_0018.88.mp4
22:02:09 | WARNING | Fin de stream o lectura fallida en frame 151
22:02:09 | INFO | Frames guardados: 11


## 4. Persistencia (metadata)


In [6]:
metadata = {
    "source": str(resolved),
    "width": width,
    "height": height,
    "fps_native": fps_native,
    "fps_sample_target": FPS_SAMPLE,
    "frame_interval": frame_interval,
    "frames_saved": saved,
    "timestamps_sec": timestamps,
    "created_at": utc_now_iso(),
}
write_json(OUT_DIR / "metadata.json", metadata)
metadata


{'source': '/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/data_sample/InHARD-master/01-InHARD/Segmented/RGBSegmented/Assemble system/P01_R01_0013.84_0018.88.mp4',
 'width': 1280,
 'height': 720,
 'fps_native': 29.0,
 'fps_sample_target': 2.0,
 'frame_interval': 14,
 'frames_saved': 11,
 'timestamps_sec': [0.012520074844360352,
  0.05900883674621582,
  0.08125495910644531,
  0.0967860221862793,
  0.1101841926574707,
  0.12346506118774414,
  0.13783597946166992,
  0.15245699882507324,
  0.16681790351867676,
  0.1798858642578125,
  0.19426798820495605],
 'created_at': '2026-05-19T04:02:09.355005+00:00'}

## 5. Validación


In [7]:
assert saved >= MIN_FRAMES_OK or saved > 0, (
    f"Se esperaban >={MIN_FRAMES_OK} frames (o al menos 1 en entorno sin cámara). "
    f"Obtuvo {saved}. Coloque un .mp4 en InHARD o ajuste MAX_FRAMES."
)
print(f"OK — {saved} frames en {OUT_DIR}")


OK — 11 frames en /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/outputs/01_capture


## 6. Siguiente paso

Ejecutar **[02_segment_frames.ipynb](02_segment_frames.ipynb)**.
